# Polish GPT-2 + LoRA: klasyfikacja tekstu i tokenów

Te same dane i podział co w `bert_peft.ipynb`. Do każdego tekstu doklejany jest `</s>`. Padding robi osobny token `<pad>`, więc ostatni token sekwencji to `</s>`. W klasyfikacji tokenów padding i `</s>` dostają etykietę -100.

In [1]:
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, GPT2ForSequenceClassification, GPT2ForTokenClassification

from lm_common import *

MODEL = "sdadas/polish-gpt2-medium"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = "<pad>"
EOS = tok.eos_token
train_df, val_df, test_df = load_data()

enc = tok(train_df.text[0] + EOS, return_offsets_mapping=True)
print(list(zip(tok.convert_ids_to_tokens(enc["input_ids"]), bio_labels(enc, train_df.spans[0], set(tok.all_special_ids))))[-5:])

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[('.', 0), ('ĠDziÄĻkujÄĻ', 0), ('Ġbardzo', 0), ('.', 0), ('</s>', -100)]


## Klasyfikacja tekstu (GPT2ForSequenceClassification)

In [2]:
lora = dict(r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["c_attn"], fan_in_fan_out=True)
seq_model = GPT2ForSequenceClassification.from_pretrained(MODEL, num_labels=2, pad_token_id=tok.pad_token_id, id2label=dict(enumerate(TON)), label2id={l: i for i, l in enumerate(TON)})
seq_model = get_peft_model(seq_model, LoraConfig(task_type=TaskType.SEQ_CLS, **lora))
seq_model.print_trainable_parameters()

seq_trainer = train(seq_model, tok, seq_dataset(train_df, tok, EOS), seq_dataset(val_df, tok, EOS), epochs=10, name="gpt2-seq")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 12186.80it/s]


[transformers] GPT2ForSequenceClassification LOAD REPORT from: sdadas/polish-gpt2-medium
Key                                     | Status     | 
----------------------------------------+------------+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED | 
score.weight                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,574,912 || all params: 358,414,336 || trainable%: 0.4394


Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map: 100%|██████████| 480/480 [00:00<00:00, 20498.77 examples/s]

... (2 outputs left out)

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:498.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.612044,0.570662,0.716667,0.417476
2,0.483360,0.264872,0.883333,0.853710
3,0.248706,0.349721,0.866667,0.850000
4,0.243709,0.202535,0.900000,0.876881
5,0.129038,0.344938,0.916667,0.895507
6,0.091360,0.481759,0.933333,0.920635
7,0.035569,0.554770,0.916667,0.902312
8,0.013965,0.685230,0.916667,0.902312
9,0.002309,0.733990,0.916667,0.902312
10,0.003331,0.739054,0.916667,0.902312


[W1103 16:30:41.570904043 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 35651584 bytes (free: 12582912, total: 17163091968).


[W1103 16:30:43.174504588 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 37748736 bytes (free: 18874368, total: 17163091968).


## Klasyfikacja tokenów (GPT2ForTokenClassification)

In [3]:
tok_model = GPT2ForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS), pad_token_id=tok.pad_token_id, id2label=dict(enumerate(TAGS)), label2id={l: i for i, l in enumerate(TAGS)})
tok_model = get_peft_model(tok_model, LoraConfig(task_type=TaskType.TOKEN_CLS, **lora))
tok_model.print_trainable_parameters()

tok_trainer = train(tok_model, tok, tok_dataset(train_df, tok, EOS), tok_dataset(val_df, tok, EOS), token_level=True, epochs=15, lr=5e-4, name="gpt2-tok")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 11953.90it/s]


[transformers] GPT2ForTokenClassification LOAD REPORT from: sdadas/polish-gpt2-medium
Key                                     | Status     | 
----------------------------------------+------------+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,577,989 || all params: 358,420,490 || trainable%: 0.4403


Epoch,Training Loss,Validation Loss,F1
1,0.553674,0.426104,0.000000
2,0.396183,0.395376,0.030534
3,0.349541,0.363294,0.010582
4,0.320344,0.371871,0.048387
5,0.290926,0.386511,0.058252
6,0.265452,0.391077,0.103030
7,0.244029,0.404016,0.079208
8,0.221174,0.416179,0.084848
9,0.203940,0.463862,0.105960
10,0.184261,0.484256,0.088737


[W1103 16:32:49.460209996 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 35651584 bytes (free: 35651584, total: 17163091968).


[W1103 16:32:51.069059782 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 37748736 bytes (free: 6291456, total: 17163091968).


## Ocena na zbiorze testowym

In [4]:
seq_preds = predict(seq_model, tok, test_df.text.tolist(), EOS)
print(classification_report(test_df.label, seq_preds, target_names=TON, digits=3))

              precision    recall  f1-score   support

      zwykły      0.886     0.907     0.897        43
   agresywny      0.750     0.706     0.727        17

    accuracy                          0.850        60
   macro avg      0.818     0.806     0.812        60
weighted avg      0.848     0.850     0.849        60



In [5]:
spans_rep, tokens_rep = report_tokens(tok_trainer, tok_dataset(test_df, tok, EOS))
print("Ocena na poziomie spanów (dokładne dopasowanie, seqeval):")
print(spans_rep)
print("Ocena na poziomie tokenów:")
print(tokens_rep)

Ocena na poziomie spanów (dokładne dopasowanie, seqeval):
              precision    recall  f1-score   support

     Agresja      0.083     0.125     0.100        32
    Kontekst      0.038     0.050     0.043       119

   micro avg      0.049     0.066     0.056       151
   macro avg      0.061     0.088     0.072       151
weighted avg      0.048     0.066     0.055       151

Ocena na poziomie tokenów:
              precision    recall  f1-score   support

     Agresja      0.379     0.363     0.371        91
    Kontekst      0.308     0.228     0.262       413
           O      0.922     0.946     0.933      4259

    accuracy                          0.872      4763
   macro avg      0.536     0.512     0.522      4763
weighted avg      0.858     0.872     0.864      4763



In [6]:
for row in test_df.head(3).itertuples():
    print(row.text)
    print("złote:", [(s["labels"][0], s["text"]) for s in row.spans])
    print("pred: ", show_spans(row.text, tok, tok_model, EOS))
    print()

To też jest dobra wiadomość. Racjonalne państwo powinno po prostu korzystać z tych okazji, zwłaszcza że Polska potrzebuje dużych inwestycji nie tylko w zbrojenia, ale także w system energetyczny. Mamy system energetyczny na trytytki. Nie zbudowaliśmy ani jednej elektrowni jądrowej i za chwilę bardzo dużo może nas to po prostu kosztować. Proszę państwa, kiedy jest szansa, żeby nadrobić te zapóźnienia, wykorzystajmy te środki. Logiczne rozwiązanie jest takie: SAFE plus atom. Bierzemy środki z SAFE na obronność, a środki z NBP przeznaczamy na budowę elektrowni jądrowych i nadrabiamy zapóźnienia.
złote: [('Kontekst', 'Mamy system energetyczny'), ('Kontekst', 'wykorzystajmy te środki'), ('Kontekst', 'SAFE plus atom')]
pred:  [('Kontekst', 'wiadomość'), ('Kontekst', 'cjonalne państwo'), ('Kontekst', 'óź'), ('Kontekst', 'FE'), ('Kontekst', 'atom'), ('Kontekst', 'zapóźnienia')]

Drugi obszar projektu to ceny transferowe i informacja TPR. Zmienia się sposób raportowania. Informacja TPR będzie m

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:438.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


## Przestrzeń wektorowa przykładów testowych (reprezentacja ostatniego tokenu)

In [7]:
emb = embed(seq_model, tok, test_df.text.tolist(), pool="last", suffix=EOS)
plot_space(emb, test_df.label, seq_preds, test_df.text, "GPT-2 LoRA - ostatni token")

In [8]:
with seq_model.disable_adapter():
    base_emb = embed(seq_model, tok, test_df.text.tolist(), pool="last", suffix=EOS)
plot_space(base_emb, test_df.label, seq_preds, test_df.text, "GPT-2 bazowy - ostatni token")